# Analyzing Results Tutorial

This tutorial covers statistical analysis, Pareto frontiers, and result comparison in Computronium.

In [ ]:
# Install if needed
!pip install -e . --quiet

## Computing Statistics

Use `comp stats` to compute summary statistics from a store:

In [ ]:
# View stats help
!comp stats --help

## Pareto Frontier Analysis

Find Pareto-optimal configurations across multiple objectives:

In [ ]:
# View pareto help
!comp pareto --help

## Statistical Comparison (Diff)

Compare two runs with effect sizes and statistical tests:

In [ ]:
# View diff help
!comp diff --help

## Advanced Analyses

Computronium provides several advanced analysis modules:

In [ ]:
# View stability analysis help
!comp stability-analysis --help

In [ ]:
# View power analysis help
!comp power-analysis --help

## Python API for Analysis

You can also use the Python API directly for custom analyses:

In [ ]:
import duckdb
import pandas as pd

# Connect to store
conn = duckdb.connect("experiment.duckdb", read_only=True)

# Query records
df = conn.execute("""
    SELECT coordinate, seed, 
           payload->>'validation_accuracy' as val_acc,
           payload->>'walltime_total' as walltime
    FROM records
    WHERE payload->>'validation_accuracy' IS NOT NULL
""").fetchdf()

print(df.head())
print(f"\nTotal records: {len(df)}")

In [ ]:
# Parse coordinates into separate columns
import re


def parse_coord(coord):
    parts = coord.split("/")
    return pd.Series({
        "substrate": parts[0] if len(parts) > 0 else None,
        "geometry": parts[1] if len(parts) > 1 else None,
        "dynamics": parts[2] if len(parts) > 2 else None,
        "plasticity": parts[3] if len(parts) > 3 else None,
        "credit": parts[4] if len(parts) > 4 else None,
        "update": parts[5] if len(parts) > 5 else None,
    })


df_parsed = df.join(df["coordinate"].apply(parse_coord))
print(df_parsed.head())

In [ ]:
# Group by axis and compute statistics
for axis in ["substrate", "geometry", "dynamics", "plasticity", "credit", "update"]:
    if axis in df_parsed.columns:
        grouped = (
            df_parsed
            .groupby(axis)["val_acc"]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        grouped["val_acc"] = grouped["val_acc"].astype(float)
        print(f"\n=== {axis.upper()} ===")
        print(grouped.sort_values("mean", ascending=False).to_string(index=False))

## Visualizing Results

Create plots with matplotlib or plotly:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Box plot of validation accuracy by plasticity type
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_parsed, x="plasticity", y="val_acc".astype(float))
plt.title("Validation Accuracy by Plasticity Type")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Exporting Data

Export results for external analysis:

In [ ]:
# Export to CSV
!comp export --store experiment.duckdb --format csv --output results.csv

# Export to JSON
!comp export --store experiment.duckdb --format json --output results.json

## Next Steps

- Explore [stability analysis](stability_analysis.ipynb)
- Learn about [reproducibility](reproducibility.ipynb)
- Check [benchmark contributions](benchmark_contributions.ipynb)